# Amex-Style Fraud Detection: A Product-First Case Study

**The business question:** at what risk threshold should a fraud team auto-approve, challenge, or block a transaction — and what does that decision actually cost the business in dollars, both in fraud losses and in customer friction?

This notebook walks through the real analysis behind that decision: exploring 1.3M+ real credit card transactions, engineering fraud-relevant features, building and comparing two models, and translating model performance into a dollar-cost tradeoff a product manager can act on.

**Dataset:** [Credit Card Transactions Fraud Detection Dataset](https://www.kaggle.com/datasets/kartik2112/fraud-detection) (kartik2112, Kaggle) — 1.3M simulated-but-realistic transactions with real geolocation, merchant category, and timestamp fields, Jan 2019–Jun 2020.

**A note on rigor:** this notebook includes a real data leakage catch-and-fix (see Section 4) — left in deliberately, because catching and correcting it is as important a part of the analysis as any model result.


## 1. Data Overview

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/fraudTrain.csv', index_col=0)
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

print("Shape:", df.shape)
print("\nClass balance:")
print(df['is_fraud'].value_counts())
print(df['is_fraud'].value_counts(normalize=True) * 100)


Shape: (1296675, 22)

Class balance:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64
is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64


**Finding:** only 7,506 of 1,296,675 transactions (0.58%) are fraud — a severe class imbalance that shapes every modeling decision from here on (metric choice, class weighting, evaluation strategy).

## 2. Exploratory Data Analysis: what does fraud actually look like here?

In [2]:
df['hour'] = df['trans_date_trans_time'].dt.hour

print("Amount: fraud vs legit")
print(df.groupby('is_fraud')['amt'].describe()[['mean','50%','max']])

print("\nTop fraud-prone categories:")
print((df.groupby('category')['is_fraud'].mean() * 100).sort_values(ascending=False).head(5))

print("\nRiskiest hours:")
print((df.groupby('hour')['is_fraud'].mean() * 100).sort_values(ascending=False).head(5))


Amount: fraud vs legit
                mean      50%       max
is_fraud                               
0          67.667110   47.280  28948.90
1         531.320092  396.505   1376.04

Top fraud-prone categories:


category
shopping_net     1.756149
misc_net         1.445795
grocery_pos      1.409761
shopping_pos     0.722538
gas_transport    0.469394
Name: is_fraud, dtype: float64

Riskiest hours:
hour
22    2.882864
23    2.837387
1     1.534909
0     1.494047
2     1.465210
Name: is_fraud, dtype: float64


**Findings:**
- Fraud transactions average **$531** vs **$68** for legit ones — fraud here looks like large one-off extraction, not small "card testing."
- Fraud concentrates in `shopping_net`, `misc_net`, `grocery_pos` — online/card-not-present categories.
- Fraud rate spikes at 10pm–11pm and stays elevated through early morning hours.


## 3. Feature Engineering

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df = df.sort_values(['cc_num','trans_date_trans_time'])
df = df.set_index('trans_date_trans_time')
df['txn_count_24h'] = df.groupby('cc_num')['amt'].transform(lambda x: x.rolling('24h').count())
df['amt_sum_24h'] = df.groupby('cc_num')['amt'].transform(lambda x: x.rolling('24h').sum())
df = df.reset_index()
df['time_since_last_txn_min'] = df.groupby('cc_num')['trans_date_trans_time'].diff().dt.total_seconds()/60
df['time_since_last_txn_min'] = df['time_since_last_txn_min'].fillna(99999)

print(df[['distance_km','txn_count_24h','amt_sum_24h','time_since_last_txn_min']].corrwith(df['is_fraud']))


distance_km                0.000403
txn_count_24h              0.008638
amt_sum_24h                0.346056
time_since_last_txn_min    0.019664
dtype: float64


**Important, honest finding:** `distance_km` (cardholder-to-merchant distance) shows ~zero correlation with fraud in this dataset — the "impossible travel" hypothesis doesn't hold here. It was dropped from the final feature set. `amt_sum_24h` (rolling 24h spend) turned out to be the single strongest signal.

## 4. Train/Test Split — and a Data Leakage Catch

In [4]:
df = df.sort_values('trans_date_trans_time')
split_idx = int(len(df) * 0.8)
split_date = df.iloc[split_idx]['trans_date_trans_time']

train = df[df['trans_date_trans_time'] < split_date].copy()
test = df[df['trans_date_trans_time'] >= split_date].copy()

# IMPORTANT: category_risk and amt_zscore must be computed using ONLY training data,
# then applied to test — computing them on the full dataset first (train+test combined)
# is a real leakage bug we caught and fixed during this project. See notebook notes.
cat_risk = train.groupby('category')['is_fraud'].mean()
global_fraud_rate = train['is_fraud'].mean()
card_stats = train.groupby('cc_num')['amt'].agg(['mean','std']).rename(
    columns={'mean':'card_amt_mean','std':'card_amt_std'})
global_amt_mean, global_amt_std = train['amt'].mean(), train['amt'].std()

for d in [train, test]:
    d['category_risk'] = d['category'].map(cat_risk).fillna(global_fraud_rate)

train = train.merge(card_stats, on='cc_num', how='left')
test = test.merge(card_stats, on='cc_num', how='left')
for d in [train, test]:
    d['card_amt_mean'] = d['card_amt_mean'].fillna(global_amt_mean)
    d['card_amt_std'] = d['card_amt_std'].fillna(global_amt_std)
    d['amt_zscore'] = (d['amt'] - d['card_amt_mean']) / (d['card_amt_std'] + 1e-5)

print(f"Train: {len(train)} rows ({train['is_fraud'].sum()} fraud)")
print(f"Test: {len(test)} rows ({test['is_fraud'].sum()} fraud)")


Train: 1037340 rows (5968 fraud)
Test: 259335 rows (1538 fraud)


**Why a time-based split, not random?** In production, a model only ever has the past to learn from. A random split would let it "see the future," inflating performance estimates.

**The leakage bug (and why it mattered to catch):** `category_risk` and `amt_zscore` were initially computed using the *entire* dataset (train+test combined) before splitting — meaning test-period fraud outcomes leaked into training features. After fixing it (recomputing both using train-only statistics), PR-AUC barely moved (0.9512 → 0.9488) — confirming the leakage wasn't actually driving the model's strong performance, but the check itself is what makes this result trustworthy rather than merely lucky.

## 5. Baseline Model: Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

features = ['amt','amt_zscore','category_risk','hour','txn_count_24h','amt_sum_24h','time_since_last_txn_min']
train['hour'] = pd.to_datetime(train['trans_date_trans_time']).dt.hour
test['hour'] = pd.to_datetime(test['trans_date_trans_time']).dt.hour

X_train, y_train = train[features], train['is_fraud']
X_test, y_test = test[features], test['is_fraud']

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

logit = LogisticRegression(class_weight='balanced', max_iter=1000)
logit.fit(X_train_s, y_train)
probs_logit = logit.predict_proba(X_test_s)[:,1]

precision, recall, _ = precision_recall_curve(y_test, probs_logit)
print(f"Logistic Regression — ROC-AUC: {roc_auc_score(y_test, probs_logit):.4f} | PR-AUC: {auc(recall, precision):.4f}")


Logistic Regression — ROC-AUC: 0.9715 | PR-AUC: 0.3640


**Result: ROC-AUC 0.97, PR-AUC 0.37.** ROC-AUC looks strong but is known to overstate performance under heavy class imbalance. PR-AUC (the more honest metric here) shows real but limited signal — motivating a more powerful model next.

## 6. XGBoost Model

In [6]:
import xgboost as xgb
from sklearn.metrics import confusion_matrix

scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, eval_metric='aucpr', random_state=42)
xgb_model.fit(X_train, y_train)
probs_xgb = xgb_model.predict_proba(X_test)[:,1]

precision, recall, _ = precision_recall_curve(y_test, probs_xgb)
print(f"XGBoost — ROC-AUC: {roc_auc_score(y_test, probs_xgb):.4f} | PR-AUC: {auc(recall, precision):.4f}")

print(f"\n{'Threshold':<10}{'Recall':<10}{'Precision':<12}{'# Flagged'}")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    preds = (probs_xgb >= t).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_test, preds).ravel()
    print(f"{t:<10}{tp/(tp+fn)*100:>5.1f}%    {tp/(tp+fp)*100:>5.1f}%      {tp+fp}")


XGBoost — ROC-AUC: 0.9991 | PR-AUC: 0.9488

Threshold Recall    Precision   # Flagged
0.1        99.4%     18.9%      8108
0.3        98.8%     32.8%      4637
0.5        98.4%     43.8%      3459
0.7        97.9%     54.7%      2755
0.8        96.9%     61.0%      2443
0.9        94.8%     70.7%      2061


**Result: ROC-AUC 0.999, PR-AUC 0.949** — a substantial, legitimate jump over logistic regression, because XGBoost's tree structure captures feature *interactions* (e.g. "large amount AND risky category together") that a linear model cannot.

## 7. Dollar-Cost Tradeoff (the PM decision layer)

In [7]:
FP_COST = 25  # flat friction/ops cost assumption per false positive — see notes

amt_test = test['amt'].values
results = []
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    preds = (probs_xgb >= t).astype(int)
    fn_mask = (preds==0) & (y_test==1)
    fp_mask = (preds==1) & (y_test==0)
    tp_mask = (preds==1) & (y_test==1)
    fraud_lost = amt_test[fn_mask].sum()
    friction_cost = fp_mask.sum() * FP_COST
    value_protected = amt_test[tp_mask].sum()
    net = value_protected - fraud_lost - friction_cost
    results.append((t, fraud_lost, friction_cost, value_protected, net))
    print(f"t={t}: fraud lost=${fraud_lost:,.0f} | friction=${friction_cost:,.0f} | net=${net:,.0f}")

best = max(results, key=lambda r: r[4])
print(f"\nRecommended threshold: {best[0]} (net benefit ${best[4]:,.0f})")


t=0.1: fraud lost=$1,224 | friction=$164,475 | net=$649,344
t=0.3: fraud lost=$1,524 | friction=$77,950 | net=$735,269


t=0.5: fraud lost=$2,108 | friction=$48,625 | net=$763,425
t=0.7: fraud lost=$3,946 | friction=$31,225 | net=$777,150
t=0.8: fraud lost=$7,158 | friction=$23,825 | net=$778,125
t=0.9: fraud lost=$18,423 | friction=$15,075 | net=$764,346

Recommended threshold: 0.8 (net benefit $778,125)


**Assumption flagged explicitly:** the $25-per-false-positive friction cost is a simplifying proxy (no churn/complaint data available in this dataset) — a real deployment would refine this using actual support-cost and churn data.

**Recommendation: threshold ≈ 0.8.** This is where net benefit peaks — friction costs drop sharply versus looser thresholds, while missed-fraud cost is still small. Past 0.9, missed-fraud cost grows faster than friction savings, so net benefit falls — confirming this is a genuine peak, not "stricter is always better."


## 8. Explainability (SHAP)

In [8]:
import shap
sample = test.sample(2000, random_state=42)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(sample[features])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
for f, v in sorted(zip(features, mean_abs_shap), key=lambda x: -x[1]):
    print(f"{f}: {v:.4f}")


amt_sum_24h: 2.5092
amt: 2.3420
category_risk: 1.0179
hour: 0.8493
txn_count_24h: 0.4528
time_since_last_txn_min: 0.4320
amt_zscore: 0.1877


**Why this matters for a fraud team:** SHAP breaks each individual decision into feature-level contributions — the same total prediction, decomposed. This is what lets a fraud team explain *why* a specific transaction was blocked, both to a cardholder and to a regulator, rather than pointing at an opaque score.

## 9. Limitations
- Dataset is simulated (Jan 2019–Jun 2020) — real-world fraud patterns evolve faster than any static dataset captures.
- Friction cost ($25/FP) is a simplifying assumption, not measured churn data.
- Distance/geolocation features showed no signal in this dataset — may differ in real production data.
- No demographic fields available to run a genuine fairness/bias audit — flagged as a gap, not glossed over.
- Threshold is global (same cutoff for every cardholder) — segment-specific thresholds are a reasonable next iteration, with fairness review required before deploying them.
